# 5. Reglas de asociación (qué se compra junto)

**Pregunta:** ¿qué familias de producto se compran juntas? Sirve para la venta cruzada y para armar ofertas por tipo de proyecto.

**Una fila = un comprobante (factura o boleta)**, con una columna por familia: verdadero si la incluye.

**Método:** algoritmo **Apriori** (librería `mlxtend`). Busca combinaciones frecuentes y genera reglas del tipo *si compra A, también compra B*.

**Métricas de cada regla:**
- **Soporte:** en qué parte de los comprobantes aparecen A y B juntas.
- **Confianza:** de los que compran A, qué parte compra también B.
- **Lift:** cuántas veces más se compran juntas de lo que cabría esperar por azar. **Lift > 1 = asociación real.** Es la métrica principal.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

from comun import *   # conexion al almacen (con, q), CORTE, guardar(), estilo(), mape(), colores

from mlxtend.frequent_patterns import apriori, association_rules

## 1. Cestas: comprobante × familia

In [ ]:
lineas = q('''
    SELECT v.serie || '-' || v.numero AS documento, p.familia
    FROM fact_ventas v
    JOIN dim_producto p        ON p.producto_key = v.producto_key
    JOIN dim_tipo_documento td ON td.tipo_documento_key = v.tipo_documento_key
    WHERE td.signo = 1
''')
cestas = pd.crosstab(lineas.documento, lineas.familia).gt(0)
print(f'{len(cestas)} comprobantes, {cestas.shape[1]} familias')
# Solo tiene sentido buscar compras conjuntas en comprobantes con 2 familias o más
cestas = cestas[cestas.sum(axis=1) >= 2]
print(f'{len(cestas)} comprobantes con 2 familias o más')
cestas.mean().sort_values(ascending=False).round(3).head(8)

## 2. Combinaciones frecuentes y reglas

Umbrales: soporte mínimo 3 %, lift mínimo 1,2 y confianza mínima 40 %. Son los habituales para un catálogo pequeño. Si se bajan salen más reglas, pero más débiles.

In [ ]:
frecuentes = apriori(cestas, min_support=0.03, use_colnames=True)
try:
    reglas = association_rules(frecuentes, metric='lift', min_threshold=1.2)
except TypeError:  # algunas versiones de mlxtend piden también el número de cestas
    reglas = association_rules(frecuentes, len(cestas), metric='lift', min_threshold=1.2)
reglas = reglas[reglas.confidence >= 0.4].copy()
reglas['si_compra'] = reglas.antecedents.apply(lambda s: ' + '.join(sorted(s)))
reglas['tambien_compra'] = reglas.consequents.apply(lambda s: ' + '.join(sorted(s)))
reglas = reglas[['si_compra', 'tambien_compra', 'support', 'confidence', 'lift']] \
    .rename(columns={'support': 'soporte', 'confidence': 'confianza'}).sort_values('lift', ascending=False).round(3)
reglas['documentos_con_ambas'] = (reglas.soporte * len(cestas)).round(0).astype(int)
guardar(reglas, 'reglas_asociacion')
reglas

In [ ]:
top = reglas.head(8).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.barh(top.si_compra.str.title() + '  →  ' + top.tambien_compra.str.title(), top.lift, color=AZUL)
for i, (l, c) in enumerate(zip(top.lift, top.confianza)):
    ax.text(l + .05, i, f'x{l:.1f}  (confianza {c:.0%})', va='center', fontsize=8)
ax.axvline(1, color='#333', lw=1, ls='--')
ax.set_xlim(0, top.lift.max() * 1.35)
estilo(ax, 'Familias que se compran juntas (lift: veces más que por azar)')
ax.grid(axis='x', color='#e5e5e5'); ax.grid(axis='y', visible=False)
guardar_figura(fig, '5_reglas_asociacion')
plt.show()

## Conclusiones (para la memoria)

- La asociación más fuerte es **puesta a tierra → servicios técnicos** (se compran juntas unas 5 veces más que por azar, con una confianza cercana al 90 %), seguida de **alquiler de equipos → servicios técnicos**. Corresponden a proyectos llave en mano: material más instalación.
- **Tableros eléctricos → cables y conductores** aparece en tres de cada cuatro compras de tableros.
- **Aplicación:** cuando un cliente pida puesta a tierra o alquiler, ofrecer el servicio de instalación; cuando pida tableros, el cable. Es la venta cruzada que se propone para el segmento de habituales (cuaderno 04).
- Las reglas son asociaciones, no causas: indican qué ofrecer, no por qué se compra.